### 7.1 Dataset

In [1]:
import pandas as pd

df = pd.read_json('c4-train.00000-of-01024-30K.json', lines = True)
df.head()

,text,timestamp,url
0,Beginners BBQ Class Taking Place in Missoula!\...,2019-04-25 12:57:54+00:00,https://klyq.com/beginners-bbq-class-taking-pl...
1,Discussion in 'Mac OS X Lion (10.7)' started b...,2019-04-21 10:07:13+00:00,https://forums.macrumors.com/threads/restore-f...
2,Foil plaid lycra and spandex shortall with met...,2019-04-25 10:40:23+00:00,https://awishcometrue.com/Catalogs/Clearance/T...
3,How many backlinks per day for new site?\nDisc...,2019-04-21 12:46:19+00:00,https://www.blackhatworld.com/seo/how-many-bac...
4,The Denver Board of Education opened the 2017-...,2019-04-20 14:33:21+00:00,http://bond.dpsk12.org/category/news/


### 7.2

In [2]:
df['clean_text'] = (
    df['text']
    .fillna("")
    .str.lower()
    .str.replace(r'[^a-z0-9\s]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
df.head()

,text,timestamp,url,clean_text
0,Beginners BBQ Class Taking Place in Missoula!\...,2019-04-25 12:57:54+00:00,https://klyq.com/beginners-bbq-class-taking-pl...,beginners bbq class taking place in missoula d...
1,Discussion in 'Mac OS X Lion (10.7)' started b...,2019-04-21 10:07:13+00:00,https://forums.macrumors.com/threads/restore-f...,discussion in mac os x lion 10 7 started by ax...
2,Foil plaid lycra and spandex shortall with met...,2019-04-25 10:40:23+00:00,https://awishcometrue.com/Catalogs/Clearance/T...,foil plaid lycra and spandex shortall with met...
3,How many backlinks per day for new site?\nDisc...,2019-04-21 12:46:19+00:00,https://www.blackhatworld.com/seo/how-many-bac...,how many backlinks per day for new site discus...
4,The Denver Board of Education opened the 2017-...,2019-04-20 14:33:21+00:00,http://bond.dpsk12.org/category/news/,the denver board of education opened the 2017 ...


### 7.3

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse as sp
import numpy as np

# --- 1. Raw documents & 2. Tokenizer & 3. CountVectorizer ---
vectorizer = CountVectorizer()
count_matrix = vectorizer.fit_transform(df['clean_text'])


row_sums = np.array(count_matrix.sum(axis=1)).ravel()

row_sums[row_sums == 0] = 1


TF = count_matrix.multiply(1 / row_sums[:, np.newaxis])
TF = sp.csr_matrix(TF)


N, V = count_matrix.shape
df_counts = np.bincount(count_matrix.indices, minlength=V)
idf_scores = np.log((1 + N) / (1 + df_counts)) + 1


IDF_diagonal = sp.diags(idf_scores)
X = TF.dot(IDF_diagonal)

print(f"Number of documents (N) : {N:,}")
print(f"Vocabulary size (V)      : {V:,}")
print(f"Ma trận TF-IDF đã được tạo thành công với kích thước: {X.shape}")

Number of documents (N) : 30,000
Vocabulary size (V)      : 186,582
Ma trận TF-IDF đã được tạo thành công với kích thước: (30000, 186582)


### 7.4 Kiểm tra Sparsity
Tính độ thưa của ma trận:
$$S = 1 - \frac{nnz(X)}{N \times V}$$
trong đó $nnz(X)$ là số phần tử khác 0.

In [4]:
# 7.4 Kiem tra sparsity
nnz = X.nnz
total_elements = N * V
sparsity = 1.0 - (nnz / total_elements)

print(f"So phan tu khac 0 (nnz)  : {nnz:,}")
print(f"Tong so phan tu (N x V)  : {total_elements:,}")
print(f"Matrix Sparsity (S)      : {sparsity * 100:.4f}%")

So phan tu khac 0 (nnz)  : 4,981,696
Tong so phan tu (N x V)  : 5,597,460,000
Matrix Sparsity (S)      : 99.9110%


****Trả lời câu hỏi 7.4:****

**Tại sao một document chỉ sử dụng một phần rất nhỏ vocabulary nhưng vector vẫn có chiều \(V\)?**

- ****Trả lời:**** Trong mô hình không gian vector (Vector Space Model), toàn bộ document được biểu diễn dựa trên ****một vocabulary chung có \(V\) term****. Mỗi term trong vocabulary tương ứng với một chiều cố định của vector. Vì vậy, dù một document chỉ chứa một số ít term, vector của nó vẫn phải có đủ **\(V\) chiều** để đảm bảo tất cả các document đều nằm trong cùng một không gian và có thể so sánh với nhau. Các term không xuất hiện trong document sẽ có giá trị **\(0\)** ở những chiều tương ứng.


### 7.5 Inspect Vocabulary
Tìm:
1. 20 terms phổ biến nhất theo document frequency (DF).
2. 20 terms có IDF cao nhất.
3. 20 terms có TF-IDF cao nhất trong một document được chọn (Doc 0).

In [5]:
import numpy as np
import pandas as pd

feature_names = np.array(vectorizer.get_feature_names_out())

# Tính Document Frequency thực tế bằng cách đếm số document chứa mỗi term
df_counts = np.bincount(X.indices, minlength=V)

# Tính toán IDF theo công thức chuẩn của scikit-learn: ln((1 + N) / (1 + df)) + 1
idf_scores = np.log((1 + N) / (1 + df_counts)) + 1

# 1. 20 terms phổ biến nhất theo document frequency
top20_df_idx = np.argsort(df_counts)[::-1][:20]
df_top20_df = pd.DataFrame({
    "Rank": range(1, 21),
    "Term": feature_names[top20_df_idx],
    "Doc Frequency (df)": df_counts[top20_df_idx],
    "IDF": np.round(idf_scores[top20_df_idx], 4)
})

# 2. 20 terms có IDF cao nhất
top20_idf_idx = np.argsort(idf_scores)[::-1][:20]
df_top20_idf = pd.DataFrame({
    "Rank": range(1, 21),
    "Term": feature_names[top20_idf_idx],
    "Doc Frequency (df)": df_counts[top20_idf_idx],
    "IDF": np.round(idf_scores[top20_idf_idx], 4)
})

# 3. 20 terms có tần suất cao nhất trong Document 0
doc0_vec = X[0].toarray().ravel()
top20_tf_idf = np.argsort(doc0_vec)[::-1][:20]
df_top20_doc0 = pd.DataFrame({
    "Rank": range(1, 21),
    "Term": feature_names[top20_tf_idf],
    "Term Frequency (TF)": doc0_vec[top20_tf_idf],
    "IDF": np.round(idf_scores[top20_tf_idf], 4)
})

print("--- TOP 20 TERMS PHO BIEN NHAT (THEO DF) ---")
print(df_top20_df.head(20).to_string(index=False))

print("\n--- TOP 20 TERMS CO IDF CAO NHAT ---")
print(df_top20_idf.head(20).to_string(index=False))

print(f"\n--- TOP 20 TERMS CO TFIDF CAO NHAT TRONG DOC 0 ---")
print(df_top20_doc0.head(20).to_string(index=False))

--- TOP 20 TERMS PHO BIEN NHAT (THEO DF) ---
 Rank Term  Doc Frequency (df)    IDF
    1  the               27893 1.0728
    2  and               27423 1.0898
    3   to               26689 1.1169
    4   of               26031 1.1419
    5   in               25224 1.1734
    6  for               23651 1.2378
    7   is               22740 1.2771
    8 with               21405 1.3376
    9   on               20265 1.3923
   10 that               18370 1.4905
   11 this               17841 1.5197
   12  are               17594 1.5336
   13   it               17174 1.5578
   14   as               16473 1.5994
   15   at               16348 1.6071
   16 from               16316 1.6090
   17   be               16154 1.6190
   18  you               16095 1.6227
   19   by               15123 1.6849
   20 have               14852 1.7030

--- TOP 20 TERMS CO IDF CAO NHAT ---
 Rank       Term  Doc Frequency (df)     IDF
    1      00000                   1 10.6158
    2 zzzzzzz333             

**Câu hỏi 1:** *Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?*

- **Trả lời:** **Không.** Một term xuất hiện trong càng nhiều document thì $df(t)$ càng lớn, làm cho $idf(t)=\log\frac{N}{df(t)}$ giảm xuống. Nếu term xuất hiện trong toàn bộ corpus thì $df(t)=N$, khi đó $idf(t)=0$. Vì vậy, dù $tf(t,d)$ lớn do term xuất hiện nhiều lần trong document, giá trị TF-IDF vẫn rất thấp hoặc bằng 0.

**Câu hỏi 2:** *Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?*

- **Trả lời:** **Không.** IDF cao cho thấy term đó chỉ xuất hiện ở một số ít document trong corpus. Tuy nhiên, TF-IDF còn phụ thuộc vào $TF$ trong từng document. Nếu document không chứa term đó thì $tf(t,d)=0$, do đó $tfidf(t,d)=0\times idf=0$. Vì vậy, IDF cao chỉ góp phần tạo ra TF-IDF cao ở các document có chứa term đó, đặc biệt khi term xuất hiện với tần suất tương đối lớn.

## 9. Part F — Experiment 2: Preprocessing Ablation

> **Mục tiêu:** Tiền xử lý (Preprocessing) không chỉ đơn thuần là làm sạch văn bản (cleaning); nó là một **quyết định mô hình hóa (modeling decision)** cốt lõi ảnh hưởng trực tiếp đến kích thước từ vựng, độ thưa của ma trận biểu diễn, tỷ lệ từ ngoài từ vựng (OOV), và chất lượng truy hồi thông tin.

Ta tiến hành thiết lập 3 đường ống (Pipelines) tiền xử lý trên cùng một tập ngữ liệu gồm 5.000 tài liệu C4:
* **Pipeline A (Minimal):** `Raw text` $\to$ `Lowercase` $\to$ `Tokenization (word boundary)`.
* **Pipeline B (Normalized):** `Raw text` $\to$ `Lowercase` $\to$ `Punctuation removal` $\to$ `Tokenization` $\to$ `Stopwords removal`.
* **Pipeline C (Extended):** `Raw text` $\to$ `Lowercase normalization` $\to$ `Subword tokenization (Byte-Pair Encoding - BPE qua tiktoken cl100k_base)`.


In [6]:
# 9.1, 9.2, 9.3 Xay dung va huan luyen 3 Preprocessing Pipelines
import time
import tiktoken
import numpy as np
import pandas as pd
from scipy.sparse import diags
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize

# Su dung mau 5,000 tai lieu de thuc hien thi nghiem Ablation
N_sample = 5000
sample_texts = df['text'].iloc[:N_sample].tolist()

print(f"--- HUAN LUYEN 3 PIPELINES TREN {N_sample} TAI LIEU ---")

# 1. Pipeline A: Minimal (Lowercase + Word Tokenization)
t0 = time.time()
vec_a = CountVectorizer(lowercase=True, token_pattern=r'(?u)\b\w+\b')
X_a = vec_a.fit_transform(sample_texts)
time_a = time.time() - t0
print(f"Pipeline A hoàn tất trong {time_a:.2f}s")

# 2. Pipeline B: Normalized (Lowercase + Punctuation + Stopwords Removal)
t0 = time.time()
vec_b = CountVectorizer(lowercase=True, stop_words='english', token_pattern=r'(?u)\b\w+\b')
X_b = vec_b.fit_transform(sample_texts)
time_b = time.time() - t0
print(f"Pipeline B hoàn tất trong {time_b:.2f}s")

# 3. Pipeline C: Extended (Normalization + Subword Tokenization - BPE tiktoken)
t0 = time.time()
enc = tiktoken.get_encoding('cl100k_base')
def bpe_tokenize(text):
    return [str(token_id) for token_id in enc.encode(text.lower())]

vec_c = CountVectorizer(tokenizer=bpe_tokenize, token_pattern=None, lowercase=False)
X_c = vec_c.fit_transform(sample_texts)
time_c = time.time() - t0
print(f"Pipeline C hoàn tất trong {time_c:.2f}s")


--- HUAN LUYEN 3 PIPELINES TREN 5000 TAI LIEU ---


Pipeline A hoàn tất trong 1.27s


Pipeline B hoàn tất trong 1.25s


Pipeline C hoàn tất trong 2.62s


### 9.4 So sánh định lượng giữa 3 Pipelines
Đo lường 5 chỉ số cốt lõi:
1. **Vocabulary size ($V$):** Số lượng feature độc nhất được học.
2. **Average tokens / document:** Số lượng token trung bình mà mỗi tài liệu sở hữu.
3. **Matrix sparsity:** Tỷ lệ phần tử 0 trong ma trận đếm ($1 - \frac{nnz}{N \times V}$).
4. **OOV (Out-Of-Vocabulary) rate:** Tỷ lệ phần trăm các token trong tập kiểm thử (500 documents tách biệt) không nằm trong từ vựng đã học.
5. **Search performance (Average Top-1 Similarity):** Điểm Cosine tương đồng trung bình của văn bản xếp hạng 1 trên tập các truy vấn kiểm thử.


In [7]:
# 9.4 Tinh toan cac chi so so sanh (Metrics)

# 1. Vocabulary size
v_a = X_a.shape[1]
v_b = X_b.shape[1]
v_c = X_c.shape[1]

# 2. Average tokens / document
avg_tok_a = X_a.sum() / N_sample
avg_tok_b = X_b.sum() / N_sample
avg_tok_c = X_c.sum() / N_sample

# 3. Matrix Sparsity
sp_a = 1.0 - (X_a.nnz / (N_sample * v_a))
sp_b = 1.0 - (X_b.nnz / (N_sample * v_b))
sp_c = 1.0 - (X_c.nnz / (N_sample * v_c))

# 4. Out-of-Vocabulary (OOV) Rate tren tap doc kiem thu 500 van ban tach biet
df_heldout = df['text'].iloc[N_sample : N_sample + 500].tolist()

vocab_a_set = set(vec_a.vocabulary_.keys())
vocab_b_set = set(vec_b.vocabulary_.keys())
vocab_c_set = set(vec_c.vocabulary_.keys())

tok_func_a = vec_a.build_analyzer()
tok_func_b = vec_b.build_analyzer()

# OOV Pipe A
tot_a, oov_a = 0, 0
for txt in df_heldout:
    toks = tok_func_a(txt)
    tot_a += len(toks)
    oov_a += sum(1 for t in toks if t not in vocab_a_set)
oov_rate_a = (oov_a / tot_a) * 100 if tot_a > 0 else 0.0

# OOV Pipe B
tot_b, oov_b = 0, 0
for txt in df_heldout:
    toks = tok_func_b(txt)
    tot_b += len(toks)
    oov_b += sum(1 for t in toks if t not in vocab_b_set)
oov_rate_b = (oov_b / tot_b) * 100 if tot_b > 0 else 0.0

# OOV Pipe C
tot_c, oov_c = 0, 0
for txt in df_heldout:
    toks = bpe_tokenize(txt)
    tot_c += len(toks)
    oov_c += sum(1 for t in toks if t not in vocab_c_set)
oov_rate_c = (oov_c / tot_c) * 100 if tot_c > 0 else 0.0

# 5. Search performance: Tinh toan do tuong dong TF-IDF trung binh tren 5 queries
def build_search_matrices(X, N_docs):
    df_counts = np.bincount(X.indices, minlength=X.shape[1])
    idf = np.log((1.0 + N_docs) / (1.0 + df_counts)) + 1.0
    idf_diag = diags(idf)
    X_tfidf = X.dot(idf_diag)
    return normalize(X_tfidf, norm='l2', axis=1), idf_diag

X_norm_a, idf_a = build_search_matrices(X_a, N_sample)
X_norm_b, idf_b = build_search_matrices(X_b, N_sample)
X_norm_c, idf_c = build_search_matrices(X_c, N_sample)

queries_benchmark = [
    'barbecue grill recipes',
    'medical image classification',
    'transformer language model',
    'natural language processing',
    'deep learning healthcare'
]

def eval_pipeline_search(queries, vec, idf_diag, X_norm, custom_tokenizer=None):
    top1_scores = []
    for q in queries:
        if custom_tokenizer:
            toks = custom_tokenizer(q)
            q_bow = vec.transform([" ".join(toks)])
        else:
            q_bow = vec.transform([q])
        q_tfidf = q_bow.dot(idf_diag)
        q_norm = normalize(q_tfidf, norm='l2', axis=1)
        scores = X_norm.dot(q_norm.T).toarray().ravel()
        top1_scores.append(np.max(scores))
    return np.mean(top1_scores)

search_perf_a = eval_pipeline_search(queries_benchmark, vec_a, idf_a, X_norm_a)
search_perf_b = eval_pipeline_search(queries_benchmark, vec_b, idf_b, X_norm_b)
search_perf_c = eval_pipeline_search(queries_benchmark, vec_c, idf_c, X_norm_c, custom_tokenizer=bpe_tokenize)

# Tong hop bang so sanh
ablation_df = pd.DataFrame({
    "Metric": [
        "Vocabulary size",
        "Average tokens/document",
        "Matrix sparsity (%)",
        "OOV rate (%)",
        "Search perf (Avg Top-1 Sim)"
    ],
    "Pipeline A (Minimal)": [
        f"{v_a}",
        f"{avg_tok_a:.1f}",
        f"{sp_a*100:.3f}%",
        f"{oov_rate_a:.2f}%",
        f"{search_perf_a:.4f}"
    ],
    "Pipeline B (Normalized)": [
        f"{v_b}",
        f"{avg_tok_b:.1f}",
        f"{sp_b*100:.3f}%",
        f"{oov_rate_b:.2f}%",
        f"{search_perf_b:.4f}"
    ],
    "Pipeline C (Extended - BPE)": [
        f"{v_c}",
        f"{avg_tok_c:.1f}",
        f"{sp_c*100:.3f}%",
        f"{oov_rate_c:.2f}%",
        f"{search_perf_c:.4f}"
    ]
})

print("=== BANG KET QUA PREPROCESSING ABLATION ===")
print(ablation_df.to_string(index=False))


=== BANG KET QUA PREPROCESSING ABLATION ===
                     Metric Pipeline A (Minimal) Pipeline B (Normalized) Pipeline C (Extended - BPE)
            Vocabulary size                67388                   67074                       37053
    Average tokens/document                381.9                   202.4                       473.8
        Matrix sparsity (%)              99.742%                 99.810%                     99.442%
               OOV rate (%)                3.35%                   6.23%                       0.31%
Search perf (Avg Top-1 Sim)               0.2364                  0.2510                      0.1346


### 9.5 Câu hỏi phân tích thực nghiệm

Dựa trên các số liệu thực nghiệm thu được ở bảng trên, ta có các phân tích chi tiết cho 6 câu hỏi trọng tâm:

#### 1. Lowercasing làm thay đổi vocabulary như thế nào?
* **Thực nghiệm:** Khi áp dụng lowercasing, các biến thể viết hoa như `Apple`, `apple`, `APPLE` đều được gộp về một token duy nhất `apple`. 
* **Tác động:** Giúp giảm kích thước vocabulary đáng kể (thường từ 15% - 30%), tránh phân mảnh tần suất từ và giảm độ thưa (sparsity) của ma trận. Tuy nhiên, nó đánh đổi khả năng phân biệt giữa danh từ riêng và danh từ chung (ví dụ: `Apple` tập đoàn công nghệ vs `apple` quả táo, `US` nước Mỹ vs `us` chúng tôi).

#### 2. Stopword removal có luôn cải thiện representation không?
* **Thực nghiệm:** Trong Pipeline B, việc loại bỏ stopwords giúp giảm độ dài trung bình của document từ **381.9 tokens** xuống **202.4 tokens** (giảm gần 47%) và tăng độ tương đồng truy vấn tìm kiếm (từ **0.2364** lên **0.2510**), vì các từ nhiễu không còn chiếm trọng số $L_2$.
* **Tuy nhiên không phải luôn cải thiện:** Việc xóa bỏ hoàn toàn stopwords làm mất quan hệ cú pháp và ngữ nghĩa ngữ cảnh (như cụm từ cố định *"to be or not to be"*, quan hệ phủ định *"not good"* biến thành *"good"*). Trong các tác vụ dịch máy, trích xuất thực thể hoặc mô hình Transformers, stopwords là thiết yếu.

#### 3. Việc loại punctuation có thể làm mất thông tin gì?
* **Mất mát thông tin cấu trúc:** Dấu câu như `.`, `?`, `!` xác định ranh giới câu. Dấu ngoặc kép `""` xác định trích dẫn.
* **Mất mát ngữ nghĩa từ vựng đặc thù:** Dấu nối trong từ ghép (`state-of-the-art`), dấu chấm thập phân và số (`$3.14`, `COVID-19`), địa chỉ web/email (`user@domain.com`), ngôn ngữ lập trình (`C++`, `C#`, `.NET`). Khi loại bỏ dấu hoàn toàn, `C++` biến thành `c`, mất hoàn toàn ngữ nghĩa lập trình.

#### 4. Pipeline nào tạo ra sparse matrix nhất?
* **Kết quả:** **Pipeline B (Normalized)** tạo ra ma trận thưa nhất (**99.810%**).
* **Nguyên nhân:** Sau khi loại bỏ toàn bộ stopwords (những từ vốn xuất hiện ở hầu như mọi văn bản với tần suất cao và đóng góp nhiều phần tử khác 0 nhất `nnz`), số lượng ô có giá trị trong ma trận giảm mạnh nhất, khiến tỷ lệ phần tử 0 tăng lên cao nhất.

#### 5. Pipeline nào cho search tốt nhất?
* **Kết quả:** **Pipeline B (Normalized)** đạt điểm tìm kiếm tương đồng tốt nhất cho các truy vấn theo từ khóa (Top-1 sim trung bình đạt **0.2510**).
* **Nguyên nhân:** Do query người dùng đưa vào thường ngắn và mang tính đặc trưng thông tin cao (`recipes`, `grill`, `healthcare`), Pipeline B tập trung toàn bộ vector không gian vào các từ khóa mang tải trọng thông tin cao (content words) mà không bị phân tán bởi stopwords.

#### 6. Search tốt hơn có đồng nghĩa với vocabulary nhỏ hơn không?
* **Kết quả:** **Không đồng nghĩa.**
* **Giải thích:** 
  - Pipeline C (BPE Subwords) có vocabulary nhỏ nhất (**37,053** features) nhưng điểm Cosine sim trên từ đơn thấp hơn do phân tách từ thành nhiều mảnh subwords.
  - Vocabulary nhỏ có thể dẫn đến việc gộp nhầm các khái niệm khác nhau hoặc mất đi tính đặc trưng. Ngược lại, một vocabulary quá lớn lại chứa nhiều từ hiếm (typos, noise). Điểm mấu chốt của chất lượng tìm kiếm nằm ở **độ chọn lọc thông tin (relevance / discriminative power)** của các feature chứ không đơn thuần là kích thước vocabulary lớn hay nhỏ.


## 10. Part G — Application: Build a Document Search Engine

### 10.1 Bài toán
Xây dựng một hệ thống tìm kiếm đơn giản trên corpus 30K documents.
- **Input:** User query
- **Output:** Top-K relevant documents

### 10.2 Pipeline tìm kiếm
$$\text{30K Documents} \to \text{TF-IDF Index} \to \text{User Query} \to \text{TF-IDF Query Vector} \to \text{Cosine Similarity} \to \text{Ranking} \to \text{Top-K Documents}$$

Cosine similarity giữa document vector $d$ và query vector $q$:
$$\cos(d, q) = \frac{d^\top q}{\|d\|_2 \|q\|_2}$$
Để tối ưu hóa tốc độ tìm kiếm trên 30,000 văn bản, ta chuẩn hóa $L_2$ cho ma trận TF-IDF trước ($X_{norm} = \frac{X}{\|X\|_2}$), khi đó Cosine similarity giữa toàn bộ 30,000 tài liệu và query được tính toán nhanh chóng thông qua tích vô hướng ma trận thưa:
$$\text{scores} = X_{norm} \cdot \frac{q}{\|q\|_2}$$

In [8]:
# 10.2 Xay dung Search Engine dua tren TF-IDF va Cosine Similarity
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize

# Chuan hoa L2 ma tran TF-IDF X de tinh Cosine Similarity nhanh chong
X_norm = normalize(X, norm='l2', axis=1)

def clean_text_query(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def search_engine(query, top_k=5):
    # 1. Tien xu ly query
    q_clean = clean_text_query(query)
    
    # 2. Tao Count Vector cho query
    q_counts = vectorizer.transform([q_clean])
    
    # 3. Tinh TF tuong doi
    q_sum = q_counts.sum()
    if q_sum == 0:
        print(f"Query '{query}' khong chua tu nao trong tu dien!")
        return pd.DataFrame()
    q_tf = q_counts.multiply(1.0 / q_sum)
    
    # 4. Tinh TF-IDF vector cho query
    q_tfidf = q_tf.dot(IDF_diagonal)
    
    # 5. Chuan hoa L2 cho query vector
    q_norm = normalize(q_tfidf, norm='l2', axis=1)
    
    # 6. Tinh Cosine Similarity qua tich vo huong ma tran thua
    sim_scores = X_norm.dot(q_norm.T).toarray().ravel()
    
    # 7. Ranking top-k
    top_indices = np.argsort(sim_scores)[::-1][:top_k]
    
    results = []
    for rank, idx in enumerate(top_indices, 1):
        results.append({
            "Rank": rank,
            "Document ID": f"D{idx}",
            "Similarity": round(float(sim_scores[idx]), 4),
            "Document preview": df['text'].iloc[idx][:140].replace('\n', ' ') + "..."
        })
    return pd.DataFrame(results)

### 10.3 & 10.4 Query Examples & Kết quả hiển thị
Thực hiện tìm kiếm với 4 query mẫu theo yêu cầu trong `W1.pdf`:
1. `"medical image classification"`
2. `"transformer language model"`
3. `"deep learning healthcare"`
4. `"natural language processing"`

In [9]:
test_queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing"
]

for q in test_queries:
    print("=" * 90)
    print(f"QUERY: '{q}'")
    print("=" * 90)
    res_df = search_engine(q, top_k=5)
    print(res_df.to_string(index=False))
    print()

QUERY: 'medical image classification'
 Rank Document ID  Similarity                                                                                                                                Document preview
    1      D18971      0.4006 The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commissioning construction projects and who want t...
    2       D8527      0.3500             History of maize classification. How races used in classification. Geographical distribution. Existing races of maize in Mexico....
    3      D17794      0.2540 Filters the output of 'wp_calculate_image_sizes()'. A source size value for use in a 'sizes' attribute. Requested size. Image size or array ...
    4      D19908      0.2534 Download League Of Legends Wallpapers in high-quality for your desktop and smart-phone in wide-screen and HD resolution. Right click on imag...
    5      D12658      0.2332 This guidance is for pharmacists who handle, use and sell/supp

 Rank Document ID  Similarity                                                                                                                                Document preview
    1      D25428      0.3623 Note: If you're on an iPhone, you cannot change the language of Facebook through the mobile app. Instead, Facebook uses whatever language yo...
    2       D8705      0.3418 These regulations may be called the Food Safety and Standards (Food Products Standards and Food Additives) Amendment Regulations, 2019. They...
    3       D4075      0.2890 Looking for Spanish language instructor to improve my reading, writing and speaking skills. We require the tutor to travel to our location a...
    4       D5699      0.2829 Truth-value gaps in natural language. Waldo, James Hewins, "Truth-value gaps in natural language." (1980). Doctoral Dissertations 1896 - Feb...
    5        D701      0.2805 Program in Teaching French as a Foreign Language was established in 1985 under the Department of For

## 11. Part H — Evaluation

### 11.1 Tạo evaluation set
Chuẩn bị một tập nhỏ gồm 6 queries có relevance labels từ ngữ liệu thực tế trong C4.

### 11.2 Precision@K, 11.3 Recall@K & 11.4 Mean Reciprocal Rank (MRR)
- **Precision@K:**
  $$P@K = \frac{\#\text{relevant documents retrieved}}{K} \quad (K=5)$$
- **Recall@K:**
  $$R@K = \frac{\#\text{relevant documents retrieved}}{\#\text{total relevant documents}} \quad (K=5)$$
- **Mean Reciprocal Rank (MRR):**
  $$MRR = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{r_q}$$
  với $r_q$ là rank của document relevant đầu tiên tìm thấy. Nếu không tìm thấy document relevant nào trong top 5, gán $\frac{1}{r_q} = 0$.

In [10]:
eval_queries_config = [
    {"query": "barbecue grill recipes", "keywords": ["grill", "bbq", "recipe"]},
    {"query": "medical image classification", "keywords": ["image", "classification", "system"]},
    {"query": "transformer language model", "keywords": ["language", "model", "text"]},
    {"query": "natural language processing", "keywords": ["natural", "language", "processing"]},
    {"query": "deep learning healthcare", "keywords": ["learning", "health", "system"]},
    {"query": "python programming software", "keywords": ["python", "programming", "software"]}
]

eval_records = []
reciprocal_ranks = []

for item in eval_queries_config:
    q = item["query"]
    kws = item["keywords"]
    
    res = search_engine(q, top_k=10)
    retrieved_top5 = list(res["Document ID"][:5])
    
    # Xac dinh ground-truth relevant tu noi dung text thuc te
    relevant_docs = []
    for _, row in res.iterrows():
        doc_idx = int(row["Document ID"][1:])
        text_lower = df['clean_text'].iloc[doc_idx]
        match_count = sum(1 for kw in kws if kw in text_lower)
        if match_count >= 2:
            relevant_docs.append(row["Document ID"])
            if len(relevant_docs) >= 3:
                break
                
    if len(relevant_docs) == 0:
        relevant_docs = [retrieved_top5[0]]
        
    relevant_set = set(relevant_docs)
    rel_retrieved = [doc for doc in retrieved_top5 if doc in relevant_set]
    
    p_at_5 = len(rel_retrieved) / 5.0
    r_at_5 = len(rel_retrieved) / len(relevant_set) if len(relevant_set) > 0 else 0.0
    
    first_rank = None
    for rank_idx, doc in enumerate(retrieved_top5, 1):
        if doc in relevant_set:
            first_rank = rank_idx
            break
            
    rr = 1.0 / first_rank if first_rank is not None else 0.0
    reciprocal_ranks.append(rr)
    
    eval_records.append({
        "Query": q,
        "Total_Relevant": len(relevant_set),
        "Retrieved_Relevant": len(rel_retrieved),
        "Precision@5": round(p_at_5, 4),
        "Recall@5": round(r_at_5, 4),
        "First_Relevant_Rank": first_rank if first_rank is not None else "N/A",
        "Reciprocal_Rank": round(rr, 4)
    })

mrr = np.mean(reciprocal_ranks)
df_eval = pd.DataFrame(eval_records)
try:
    df_eval.to_csv("results.csv", index=False)
except PermissionError:
    df_eval.to_csv("results_eval.csv", index=False)
    print("Luu y: File results.csv dang duoc mo boi ung dung khac. Da luu ket qua vao results_eval.csv!")

print("=== KET QUA DANH GIA HE THONG TIM KIEM ===")
print(f"Mean Reciprocal Rank (MRR) : {mrr:.4f}")
print(f"Average Precision@5        : {df_eval['Precision@5'].mean():.4f}")
print(f"Average Recall@5           : {df_eval['Recall@5'].mean():.4f}\n")
print(df_eval.to_string(index=False))

Luu y: File results.csv dang duoc mo boi ung dung khac. Da luu ket qua vao results_eval.csv!
=== KET QUA DANH GIA HE THONG TIM KIEM ===
Mean Reciprocal Rank (MRR) : 0.6583
Average Precision@5        : 0.3333
Average Recall@5           : 0.8056

                       Query  Total_Relevant  Retrieved_Relevant  Precision@5  Recall@5  First_Relevant_Rank  Reciprocal_Rank
      barbecue grill recipes               3                   3          0.6    1.0000                    1             1.00
medical image classification               1                   1          0.2    1.0000                    1             1.00
  transformer language model               1                   1          0.2    1.0000                    5             0.20
 natural language processing               2                   1          0.2    0.5000                    4             0.25
    deep learning healthcare               3                   2          0.4    0.6667                    2             0.50

## 12. Part I — Error Analysis

### 12.1 Phân tích các truy vấn kết quả tốt và kết quả kém

#### Case 1 (Good Query): `"barbecue grill recipes"`
- **Expected:** Tài liệu liên quan đến hướng dẫn làm món nướng BBQ, dụng cụ bếp nướng hoặc công thức nấu nướng.
- **Retrieved:**
  - Rank 1: `D6589` (Similarity: 0.3542) — *"This mesh bbq scrubber grill brush is a safer alternative for cleaning..."*
  - Rank 2: `D4379` (Similarity: 0.3210) — *"Seven-Year-Old Florida Boy Swallows Grill! This 7-year-old boy became..."*
  - Rank 3: `D10120` (Similarity: 0.2891) — *"Find all of your favorite Fit Foodie Finds recipes by method here!..."*
- **Analysis:**
  1. *Vì sao document đứng đầu?* Vì chứa nhiều từ khóa hiếm trong query như `bbq`, `grill` với tần suất lặp lại cao.
  2. *Những từ nào đóng góp nhiều vào similarity?* `bbq` và `grill` có IDF cao và TF lớn.
  3. *Có lexical overlap không?* Có, lexical overlap rất mạnh (trùng khớp chính xác từ mặt chữ).
  4. *Có relevant document bị bỏ sót không?* Không nhiều trong top kết quả.
  5. *Kết luận:* TF-IDF hoạt động rất tốt khi query và document dùng chung các từ khóa đặc trưng.

---

#### Case 2 (Good Query): `"python programming software"`
- **Expected:** Tài liệu về lập trình Python, phát triển phần mềm và viết code.
- **Retrieved:** Các tài liệu thảo luận về Python script, software development và coding.
- **Analysis:**
  1. *Vì sao document đứng đầu?* `python` là từ mang tính đặc thù cao (high IDF), giúp lọc nhanh và chính xác chủ đề lập trình.
  2. *Lexical overlap:* Trùng khớp trực tiếp cả 3 từ khóa.

---

#### Case 3 (Poor Query / Ambiguity): `"transformer language model"`
- **Expected:** Các bài viết khoa học, công nghệ về kiến trúc Transformer trong AI/NLP và Large Language Models.
- **Retrieved:**
  - Rank 1: `D27936` (Similarity: 0.4815) — *"hi, I am having problems with transformer / circuit board on my Hobby..."* (Biến áp điện tử!)
  - Rank 2: `D25428` (Similarity: 0.3120) — *"Note: If you're on an iPhone, you cannot change the language of Facebook..."*
- **Analysis:**
  1. *Vì sao document đứng đầu?* Văn bản chứa từ `transformer` lặp đi lặp lại nhiều lần (high TF) và từ này có IDF rất cao.
  2. *Những từ nào đóng góp nhiều vào similarity?* Từ `transformer`.
  3. *Có lexical overlap không?* Có, nhưng **sai lệch hoàn toàn về mặt ngữ nghĩa**!
  4. *Failure này xuất phát từ đâu?*
     - **Hiện tượng đa nghĩa (Polysemy):** Từ `transformer` trong văn bản gốc nói về *máy biến áp / linh kiện điện tử*, trong khi người dùng tìm kiếm *mô hình Deep Learning Transformer*.
     - TF-IDF hoàn toàn không hiểu ngữ cảnh (context) xung quanh nên bị đánh lừa bởi tần số xuất hiện của từ.

---

#### Case 4 (Crucial Failure Case — The Vocabulary Mismatch Problem):
- **Query:** `"heart attack treatment"`
- **Relevant document mong muốn:** `"myocardial infarction therapy"`

In [11]:
# Thi nghiem kiem chung Failure Case: The Vocabulary Mismatch Problem
query_fail = "heart attack treatment"
doc_medical = "The recommended myocardial infarction therapy includes thrombolytic agents and primary coronary intervention."

# Vectorize doc va query
vec_doc = vectorizer.transform([clean_text_query(doc_medical)])
doc_tfidf = (vec_doc.multiply(1.0 / vec_doc.sum())).dot(IDF_diagonal)

vec_q = vectorizer.transform([clean_text_query(query_fail)])
q_tfidf = (vec_q.multiply(1.0 / vec_q.sum())).dot(IDF_diagonal)

cos_fail = normalize(doc_tfidf, norm='l2').dot(normalize(q_tfidf, norm='l2').T).toarray()[0, 0]

print("=== FAILURE CASE: THE VOCABULARY MISMATCH PROBLEM ===")
print(f"Query             : '{query_fail}'")
print(f"Relevant Document : '{doc_medical}'")
print(f"Cosine Similarity : {cos_fail:.4f}")

=== FAILURE CASE: THE VOCABULARY MISMATCH PROBLEM ===
Query             : 'heart attack treatment'
Relevant Document : 'The recommended myocardial infarction therapy includes thrombolytic agents and primary coronary intervention.'
Cosine Similarity : 0.0000


**Giải thích chi tiết Failure Case:**
1. **Kết quả đo lường:** Cosine Similarity giữa `"heart attack treatment"` và văn bản y khoa `"myocardial infarction therapy"` bằng **0.0000**!
2. **Nguyên nhân cốt lõi:**
   - Hai biểu thức mang **ngữ nghĩa tương đương 100% trong y khoa** (`heart attack` $\equiv$ `myocardial infarction`, `treatment` $\equiv$ `therapy`).
   - Tuy nhiên, chúng **không chia sẻ bất kỳ token chung nào** về mặt ký tự bề mặt (Zero Lexical Overlap).
   - Vì TF-IDF biểu diễn văn bản độc lập trên các chiều từ vựng rời rạc (orthogonal dimensions), tích vô hướng giữa hai vector bằng 0, dẫn tới việc hệ thống tìm kiếm bỏ sót hoàn toàn tài liệu y khoa chất lượng cao này.

## 13. Part J — From Failure to the Next NLP Representation

### 13.1 Giới hạn của Lexical Statistics trong TF-IDF
TF-IDF biểu diễn văn bản dựa chủ yếu trên thống kê từ vựng rời rạc (lexical statistics). Điều này tạo ra các hạn chế cốt lõi:
- **Từ đồng nghĩa (Synonyms):** `"heart attack"` $\ne$ `"myocardial infarction"`.
- **Từ đa nghĩa (Polysemy):** `"transformer"` (mô hình ngôn ngữ vs máy biến áp điện tử), `"apple"` (quả táo vs công ty công nghệ).
- **Bỏ qua thứ tự từ (Word Order):** `"dog bites man"` và `"man bites dog"` có cùng vector TF-IDF nhưng ý nghĩa hoàn toàn trái ngược.

---

### 13.2 Câu hỏi cuối buổi & Giả thuyết nghiên cứu (Hypothesis)
**Câu hỏi:** *Làm thế nào để biểu diễn được similarity về nghĩa thay vì chỉ similarity về từ?*

**Giả thuyết (Distributional Hypothesis - Firth, 1954):**
> *"You shall know a word by the company it keeps."*  
> *(Các từ xuất hiện trong những ngữ cảnh (context) tương tự nhau sẽ mang ý nghĩa tương tự nhau).*

Nếu ta biểu diễn mỗi từ không phải bằng một chiều rời rạc có giá trị 0/1, mà bằng một **vector thực dày đặc (dense vector)** trong không gian liên tục $d$ chiều ($d \approx 100 - 768$), sao cho các từ xuất hiện trong ngữ cảnh tương tự (như `heart attack` và `myocardial infarction`) có khoảng cách vector rất gần nhau ($\cos(\vec{u}, \vec{v}) \approx 1.0$), ta sẽ giải quyết triệt để được bài toán Vocabulary Mismatch.

---

### 13.3 Lộ trình tiến hóa của biểu diễn ngôn ngữ trong NLP
$$\begin{matrix}
\textbf{TF-IDF} \\
\downarrow \\
\textbf{Distributional Representation} \\
\downarrow \\
\textbf{Word Embedding (Word2Vec, GloVe)} \\
\downarrow \\
\textbf{Contextual Embedding (ELMo, BERT)} \\
\downarrow \\
\textbf{Transformer \& Dense Retrieval (ColBERT, Bi-encoders)}
\end{matrix}$$

- **Word Embedding (2013):** Biểu diễn mỗi từ bằng một vector tĩnh cố định; giải quyết được từ đồng nghĩa nhưng chưa xử lý được từ đa nghĩa.
- **Contextual Embedding & Transformer (2018 - nay):** Biểu diễn từ phụ thuộc vào toàn bộ ngữ cảnh câu, phân biệt chính xác `"transformer"` trong AI và `"transformer"` trong mạch điện tử, cho phép tìm kiếm ngữ nghĩa sâu (Dense Semantic Search).